In [6]:
import sys
sys.path.append("..")

import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

df = pd.read_csv("../data/clean.csv")
df["clean"] = df["clean"].fillna("")

X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label_num"], test_size=0.2, random_state=42, stratify=df["label_num"]
)
print(len(X_train), len(X_test))

4102 1026


In [7]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC()),
])

params = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [3000, 5000],
    "clf__C": [0.1, 0.5, 1, 2],
}

grid = GridSearchCV(pipe, params, scoring="f1", cv=5)
grid.fit(X_train, y_train)

print("Best settings:", grid.best_params_)
print("Best cross-validation F1:", round(grid.best_score_, 4))

Best settings: {'clf__C': 2, 'tfidf__max_features': 3000, 'tfidf__ngram_range': (1, 1)}
Best cross-validation F1: 0.9131


In [8]:
best = grid.best_estimator_
y_pred = best.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       896
        spam       0.98      0.90      0.94       130

    accuracy                           0.99      1026
   macro avg       0.98      0.95      0.97      1026
weighted avg       0.99      0.99      0.99      1026



In [9]:
from sklearn.calibration import CalibratedClassifierCV
from preprocess import clean_text

bp = grid.best_params_

final = Pipeline([
    ("tfidf", TfidfVectorizer(
        preprocessor=clean_text,
        ngram_range=bp["tfidf__ngram_range"],
        max_features=bp["tfidf__max_features"],
    )),
    ("clf", CalibratedClassifierCV(LinearSVC(C=bp["clf__C"]), cv=5)),
])

# same split, but on the RAW messages
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    df["message"], df["label_num"], test_size=0.2, random_state=42, stratify=df["label_num"]
)

final.fit(Xr_train, yr_train)
print(classification_report(yr_test, final.predict(Xr_test), target_names=["ham", "spam"]))

os.makedirs("../models", exist_ok=True)
joblib.dump(final, "../models/spam_model.joblib")
print("Saved!")

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       896
        spam       0.98      0.91      0.94       130

    accuracy                           0.99      1026
   macro avg       0.99      0.95      0.97      1026
weighted avg       0.99      0.99      0.99      1026

Saved!


In [10]:
model = joblib.load("../models/spam_model.joblib")

tests = [
    "WINNER!! You have won a £900 prize. Call 09061701461 to claim now",
    "Hey, are we meeting for lunch tomorrow?",
    "Train tickets (and more!) in the Uber app. Book your next journey today.",
]

for t in tests:
    p = model.predict_proba([t])[0][1]
    print("SPAM" if p > 0.5 else "HAM", f"({p:.2f})", "-", t)

SPAM (1.00) - WINNER!! You have won a £900 prize. Call 09061701461 to claim now
HAM (0.00) - Hey, are we meeting for lunch tomorrow?
HAM (0.01) - Train tickets (and more!) in the Uber app. Book your next journey today.
